# California Housing Exploratory Analysis and Preprocessing

This notebook documents the exploratory analysis and preprocessing used for a 2025 California single-family-home price project. January through August form the training period; September and October form a later-time held-out period.

Raw MLS records are not included in the public repository. Retained outputs contain aggregate summaries and visualizations only. The final model build is documented in `modeling.ipynb`.

## Libraries

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from sklearn.linear_model import LinearRegression

Could not save font_manager cache [Errno 13] Permission denied: 'C:\\Users\\mmiov\\.matplotlib\\fontlist-v3.11.0.json.matplotlib-lock'


## Load, verify, and merge monthly data

The chronological split keeps January through August 2025 in the training set and reserves September and October 2025 for held-out evaluation. All preprocessing decisions are learned from the training data and then applied to the held-out months.

### Load Housing Data

In [2]:
# January through August training months
df1 = pd.read_csv('CRMLSSold202503_filled.csv')    # March
df2 = pd.read_csv('CRMLSSold202504_filled.csv')    # April
df3 = pd.read_csv('CRMLSSold202505_filled.csv')    # May
df4 = pd.read_csv('CRMLSSold202506_filled.csv')    # June
df5 = pd.read_csv('CRMLSSold202507_filled.csv')    # July
df6 = pd.read_csv('CRMLSSold202508_filled-2.csv')  # August
df7 = pd.read_csv('CRMLSSold202501_filled.csv')    # January
df8 = pd.read_csv('CRMLSSold202502_filled.csv')    # February

# September and October held-out months
tst_sep = pd.read_csv('CRMLSSold202509.csv')
tst_oct = pd.read_csv('CRMLSSold202510.csv')

C:\Users\mmiov\AppData\Local\Temp\ipykernel_84732\2234052275.py:5: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df4 = pd.read_csv('CRMLSSold202506_filled.csv')    # June


### Verify Housing Data Types and Structure

Matching schemas are required before concatenating the monthly training files.

In [3]:
training_months = [df7, df8, df1, df2, df3, df4, df5, df6]
held_out_months = [tst_sep, tst_oct]

training_columns = [list(frame.columns) for frame in training_months]
all_equal = all(training_columns[0] == columns for columns in training_columns[1:])
print('Training-month columns identical?', all_equal)

if not all_equal:
    for month_number, columns in enumerate(training_columns, start=1):
        print(f'Training frame {month_number} columns: {columns}')

Training-month columns identical? True


`training_months` and `held_out_months` keep the chronological roles explicit. They are processed with the same feature logic, but the groups are never mixed.

The eight training files share an exact schema. The September and October files omit `LonFilled` and `LatFilled`; neither field is retained for analysis or modeling, so the difference does not affect the final feature contract.

In [4]:
trn = pd.concat(training_months, ignore_index=True)
tst = pd.concat(held_out_months, ignore_index=True)

print('Training data')
trn.info()
print()
print('Held-out data')
tst.info()

Training data


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 174802 entries, 0 to 174801
Data columns (total 80 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   BuyerAgentAOR                 174726 non-null  object 
 1   ListAgentAOR                  174726 non-null  object 
 2   Flooring                      102766 non-null  object 
 3   ViewYN                        157593 non-null  object 
 4   WaterfrontYN                  108 non-null     object 
 5   BasementYN                    2870 non-null    object 
 6   PoolPrivateYN                 154976 non-null  object 
 7   OriginalListPrice             174263 non-null  float64
 8   ListingKey                    174802 non-null  int64  
 9   ListAgentEmail                174151 non-null  object 
 10  CloseDate                     174802 non-null  object 
 11  ClosePrice                    174800 non-null  float64
 12  ListAgentFirstName            173918 non-nul

## Pre-Defined Filters

Filter the records to:

*   `PropertyType = Residential`
*   `PropertySubType = SingleFamilyResidence`


*   `StateOrProvince = CA`

`PropertyType=Residential` defines the broad residential category. `PropertySubType=SingleFamilyResidence` removes condos, townhouses, multifamily properties, and mobile homes, leaving detached houses. `StateOrProvince=CA` restricts the target population to California.

In [5]:
# pre-filter count
n1 = len(trn)
n2 = len(tst)
print(f'The size of the training data before filtering is: ', n1)
print(f'The size of the testing data before filtering is: ', n2)

The size of the training data before filtering is:  174802
The size of the testing data before filtering is:  45676


In [6]:
# applying filters to train and test sets
trn = trn[(trn['PropertyType'] == 'Residential') & (trn['PropertySubType'] == 'SingleFamilyResidence') & (trn['StateOrProvince'] == 'CA')]
tst = tst[(tst['PropertyType'] == 'Residential') & (tst['PropertySubType'] == 'SingleFamilyResidence') & (tst['StateOrProvince'] == 'CA')]

In [7]:
# Count of dropped instances
print(f'Filtering dropped' , n1 - len(trn), 'instances from the training data')
print(f'Filtering dropped' , n2 - len(tst), 'instances from the testing data')

# new sizes of train and test
print(f'The size of the training data after filtering is: ', len(trn))
print(f'The size of the testing data after filtering is: ', len(tst))

Filtering dropped 88279 instances from the training data
Filtering dropped 22192 instances from the testing data
The size of the training data after filtering is:  86523
The size of the testing data after filtering is:  23484


## Missing Values

### Missingness Percentage by Feature for Training



In [8]:
# show all cols and rows
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

# calculate percentage of missing based on sample size
missing_percentages = trn.isna().sum() / len(trn) * 100
missing_percentages = missing_percentages.sort_values(ascending = False)
print(missing_percentages)

# revert back to original pd settings
pd.reset_option("display.max_columns")
pd.reset_option("display.max_rows")

TaxYear                         100.000000
ElementarySchoolDistrict        100.000000
BusinessType                    100.000000
CoveredSpaces                   100.000000
MiddleOrJuniorSchoolDistrict    100.000000
TaxAnnualAmount                 100.000000
AboveGradeFinishedArea          100.000000
FireplacesTotal                 100.000000
WaterfrontYN                     99.946835
BelowGradeFinishedArea           99.325035
BasementYN                       97.577523
BuilderName                      95.306450
LotSizeDimensions                93.745016
BuildingAreaTotal                93.352057
CoBuyerAgentFirstName            90.974654
ElementarySchool                 86.570045
MiddleOrJuniorSchool             86.453313
HighSchool                       82.423171
CoListAgentFirstName             76.940236
CoListOfficeName                 76.875513
CoListAgentLastName              76.874357
AssociationFeeFrequency          74.486553
SubdivisionName                  64.724986
MainLevelBe

#### Initial Interpretation

`TaxYear`, Integer, 100.000000, Remove; Highly missing

`ElementarySchoolDistrict`, String, 100.000000, Remove; Highly missing

`BusinessType`, String, 100.000000, Remove; Highly missing

`CoveredSpaces`, Integer, 100.000000, Remove; Highly missing

`MiddleOrJuniorSchoolDistrict`, String, 100.000000, Remove; Highly missing

`TaxAnnualAmount`, Decimal, 100.000000, Remove; Highly missing

`AboveGradeFinishedArea`, Decimal, 100.000000, Remove; Highly missing

`FireplacesTotal`, Integer, 100.000000, Remove; Highly missing

`WaterfrontYN`, Bool, 99.942725, Remove; Highly missing

`BelowGradeFinishedArea`, Decimal, 99.336681, Remove; Highly missing

`BasementYN`, Bool, 97.559839, Remove; Highly missing

`BuilderName`, String, 95.259534, Remove; Highly missing

`LotSizeDimensions`, String, 93.770396, Remove; Highly missing

`BuildingAreaTotal`, Decimal, 93.366810, Remove; Highly missing

`CoBuyerAgentFirstName`, String, 90.986587, Remove; Highly missing

`ElementarySchool`, String, 86.568456, Remove; Highly missing

`MiddleOrJuniorSchool`, String, 86.429932, Remove; Highly missing

`HighSchool`, String, 82.390080, Remove; Highly missing

`CoListAgentFirstName`, String, 76.942339, Remove; Highly missing

`CoListAgentLastName`, String, 76.871745, Remove; Highly missing

`CoListOfficeName`, String, 76.870413, Remove; Highly missing

`AssociationFeeFrequency`, String, 74.464883, Remove; Highly missing

`SubdivisionName`, String, 64.768171, Remove; Highly missing

`MainLevelBedrooms`, Integer, 39.665943, Remove; Highly missing

`Flooring`, Categorical, 35.444943, Type of flooring, Maybe Keep; depending on if missingness is too high and on how dirty it is

`AssociationFee`, Decimal, 29.529683, HOA fee, Maybe Keep; depending on if missingness is too high

`HighSchoolDistrict`, String, 26.860423, Remove; Likely not predictive of house price

`MLSAreaMajor`, String, 14.301317, the minor/sub marketing area name, Remove; not predictive of house price

`AttachedGarageYN`, Bool, 11.819865, Is the garage attached to the property, Keep; lowly missing and maybe predictive of house price.

`Stories`, Integer, 11.497529, number of floors in the property being sold,  Keep; really dirty but lowly missing and likely predictive of house price.

`ViewYN`, Bool, 8.949479, Does the property have a view, Maybe Remove; feels too subjective.

`PoolPrivateYN`, Bool, 8.202246, does the property have a private pool, Keep; lowly missing and likely predictive of house price.

`Levels`, Levels Enum, 8.042410, Number of levels in the property, Maybe; likely predictive of house price but quite dirty

`NewConstructionYN`, Bool, 7.425710, Is the property newly constructed and has not been previously occupied, Keep; because not been previously occupied is possibly predictive of house price.

`GarageSpaces`, Decimal, 3.937291, The number of spaces in the garage, Keep; physical feature, likely predictive of price.

`BuyerOfficeAOR`, String, 3.788111, The buyer’s office’s board or association of realtors, Remove; likely not predictive of house price.

`LotSizeSquareFeet`, Decimal, 1.760859, Total lotsize in squarefeet, Keep; lowly missing, physical feature, likely predictive of house price, and is on a standardized scale.

`LotSizeAcres`, Decimal, 1.758195, Total acres of the lot, Keep; lowly missing, physical feature, likely predictive of house price, and is on a standardized scale

`LotSizeArea`, Decimal, 1.752867, Total area of the lot, Keep; lowly missing, physical feature, likely predictive of house price but is not on a standardized scale

`BuyerOfficeName`, String, 1.351945, Remove; not necessary for house price prediction

`ListAgentFirstName`, String, 0.723257, Remove; not necessary for house price prediction

`BuyerAgentFirstName`, String, 0.475512, Remove; not necessary for house price prediction

`ListAgentEmail`, String, 0.338319, Remove; not necessary for house price prediction

`OriginalListPrice`, Decimal, 0.201127, Keep; Although it may be highly correlated with ListPrice

`BuyerAgentMlsId`, String, 0.134529, Buyer agent MLS identification, Remove; not predictive of house price

`StreetNumberNumeric`, Integer, 0.127869, the integer portion of the street number, Remove; not a useful predictor

`UnparsedAddress`, String, 0.109221, Text representation of the address with the full civic location and may OPTIONALLY include any of the City, StateOrProvince, PostcalCode, County, Remove; structured location features are already available.

`City`, String, 0.087910, City in the listing address, Keep; physical location and could be made into a categorical although there are about 898 unique cities.

`YearBuilt`, Integer, 0.081250, Year the house was built, Keep; can be used to compute age of the house.

`FireplaceYN`, Bool, 0.061270, Does the property include a fireplace, Keep; physical feature.

`LivingArea`, Decimal, 0.055943, Total livable area in the house, Keep; physical feature likely to predict price; units require standardization.

`BuyerAgentLastName`, String, 0.025307, Buyer agent last name, Remove; likely not predictive of house price.

`BathroomsTotalInteger`, Integer, 0.019979, simple sum of number of bathrooms (rounded up, e.g., 2.5 maps to 3), Keep; physical feature so good predictor of house price.

`ListAgentLastName`, String, 0.014652, Listing agent’s last name, Remove; likely not predictive of house price.

`ListAgentFullName`, String, 0.011988, Listing agent’s name, Remove; likely not predictive of house price.

`BuyerAgentAOR`, AOR Enum, 0.010656, the Buyer’s Agent’s Board or Association of Realtors, Remove; likely not predictive of house price.

`ListAgentAOR`, AOR Enum, 0.010656, the Co Listing Agent’s Board or Association of Realtors, Remove; likely not predictive of house price.

`Latitude`, Decimal, 0.007992, Location variable, Keep; Location is important for house price.

`Longitude`, Decimal, 0.007992, Location variable, Keep; Location is important for house price.

`PurchaseContractDate`, DateTime, 0.003996, date offer is accepted and listing is no longer on market, Remove; used only to calculate `DaysOnMarket`, which is already available

`ParkingTotal`, Decimal, 0.001332, total number of parking spaces included, Keep; lowly missing and a physical feature.

`CloseDate`, DateTime, 0.000000, day house is taken off market, Keep; important for seasonal trends.

`ClosePrice`, String, 0.000000, Keep; dependent variable.

`ListingKey`, String, 0.000000, Keep; the metadata labels this feature as the primary key for the datasets.

`PropertySubType`, String, 0.000000, subtypes to the PropertyType variable, Remove; constant after filtering on `PropertySubType = SingleFamilyResidence`.

`ListingKeyNumeric`, Integral, 0.000000, Remove; redundant because `ListingKey` is the primary key.

`MlsStatus`, String, 0.000000, status of listing, Remove; only one value and it is Closed (obviously) so it is not useful.

`CountyOrParish`, String, 0.000000, county or parish or other regional authority for instance, Keep; only 57 unique values so it can be used as a categorical variable if needed.

`DaysOnMarket`, Integer, 0.000000, number of days listing is on the market, Keep; good predictor of house price.

`ListOfficeName`, String, 0.000000, legal name of the broker, Remove; too many unique values, 8742, to be considered as a categorical.

`PropertyType`, String, 0.000000, the type of property, Remove; constant after filtering on `PropertyType = Residential`.

`ListPrice`, Decimal, 0.000000, the current price of the property as determined by the seller and the seller’s broker, Keep; sets the relative price.

`StateOrProvince`, String, 0.000000, state the listing is in, Remove; constant after filtering on `StateOrProvince = CA`.

`ListingContractDate`, DateTime, 0.000000, date the listing agreement was signed between the seller and the listing agent i.e., the date the house is put on the market, Remove; redundant with `DaysOnMarket`

`BedroomsTotal`, Integer, 0.000000, total number of bedrooms in the dwelling, Keep; physical feature so good predictor.

`ContractStatusChangeDate`, DateTime, 0.000000, date when the listing’s contractual status changed (e.g., from active → pending, pending → closed, etc), Keep; useful supporting feature, captures seasonality like fall, winter, spring, summer listing

`ListingId`, String, 0.000000, Remove; unique key for a specific listing, but `ListingKey` is already the primary key.

`PostalCode`, String, 0.000000, Remove; other, more informative, location variables are available

`Latfilled`, Boolean, 0.000000, Remove; redundant latitude-completeness flag.

`Lonfilled`, Boolean, 0.000000, Remove; redundant longitude-completeness flag.

`Maybe Keep` identifies potentially useful fields whose missingness or inconsistency complicates modeling. They are excluded from the current feature set and can be reconsidered if later experiments justify the additional processing.

#### Initial Feature Drop

In [9]:
retained = [# --- Keep ---
            'AttachedGarageYN',
            'PoolPrivateYN',
            'NewConstructionYN',
            'GarageSpaces',
            'LotSizeSquareFeet',
            'LotSizeAcres',
            'LotSizeArea',
            'YearBuilt',
            'FireplaceYN',
            'LivingArea',
            'BathroomsTotalInteger',
            'Latitude',
            'Longitude',
            'ParkingTotal',
            'CloseDate',
            'ClosePrice',
            'ListingKey',
            'CountyOrParish',
            'DaysOnMarket',
            'ListPrice',
            'BedroomsTotal',
            'ContractStatusChangeDate',
            'City',
            'OriginalListPrice',
            # --- Maybe Keep ---
            'Flooring',
            'AssociationFee',
            'Levels']

# keeping only retained cols in tst and trn
tst = tst[retained].copy()
trn = trn[retained].copy()


#### Why "Maybe Keep" Instead of "Keep"

In [10]:
# far too many unique, convoluted values
print(trn['Flooring'].unique())

['Carpet,Tile' nan 'Tile' 'Stone' 'Wood' 'Vinyl' 'Carpet,Vinyl'
 'Tile,Wood' 'Carpet,Stone,Wood' 'Carpet,Laminate' 'Laminate,Wood'
 'Carpet,Wood' 'Carpet' 'Stone,Wood' 'Laminate' 'Stone,Tile,Wood'
 'Carpet,Laminate,Tile' 'Carpet,Tile,Wood' 'SeeRemarks'
 'Carpet,Stone,Tile' 'Carpet,Vinyl,Wood' 'Concrete,Laminate'
 'Laminate,Tile' 'Carpet,Tile,Vinyl' 'Vinyl,Wood' 'Stone,Vinyl'
 'Carpet,Laminate,Stone' 'SeeRemarks,Tile,Vinyl' 'Concrete'
 'Concrete,Tile' 'Carpet,Stone' 'Carpet,Laminate,Wood'
 'Carpet,Laminate,Tile,Vinyl' 'SeeRemarks,Wood'
 'Carpet,Laminate,Tile,Wood' 'Laminate,Tile,Wood' 'Carpet,Stone,Vinyl'
 'Tile,Vinyl' 'Concrete,SeeRemarks,Vinyl' 'Carpet,Laminate,Stone,Vinyl'
 'Brick,Carpet,Stone,Tile' 'Concrete,Vinyl' 'SeeRemarks,Tile,Wood'
 'Carpet,Laminate,Stone,Wood' 'Concrete,Laminate,Tile'
 'Bamboo,Carpet,Laminate,Tile' 'Carpet,Stone,Tile,Wood' 'Laminate,Vinyl'
 'Tile,Vinyl,Wood' 'SeeRemarks,Vinyl' 'Carpet,Laminate,Vinyl'
 'Carpet,SeeRemarks,Tile,Wood' 'Carpet,Laminate,Vinyl,Wood'

`AssociationFee` is nearly 30% missing, so extensive imputation could distort its distribution.

In [11]:
# also very dirty unique values
print(trn['Levels'].unique())

['One' 'Two' nan 'ThreeOrMore' 'MultiSplit' 'One,Two'
 'ThreeOrMore,MultiSplit' 'Two,MultiSplit' 'One,MultiSplit'
 'One,Two,MultiSplit' 'Two,ThreeOrMore' 'One,ThreeOrMore'
 'Two,ThreeOrMore,MultiSplit' 'One,Two,ThreeOrMore,MultiSplit'
 'One,Two,ThreeOrMore' 'Two,One' 'MultiSplit,One' 'ThreeOrMore,One']


`Flooring`, `AssociationFee`, and `Levels` are excluded from the current feature set.

In [12]:
# drop flooring, associationfee, levels from tst and trn
tst.drop(['Flooring', 'AssociationFee', 'Levels'], axis = 1, inplace = True)
trn.drop(['Flooring', 'AssociationFee', 'Levels'], axis = 1, inplace = True)

In [13]:
tst.info()
trn.info()

<class 'pandas.core.frame.DataFrame'>
Index: 23484 entries, 3 to 45668
Data columns (total 24 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   AttachedGarageYN          20645 non-null  object 
 1   PoolPrivateYN             21709 non-null  object 
 2   NewConstructionYN         21673 non-null  object 
 3   GarageSpaces              22553 non-null  float64
 4   LotSizeSquareFeet         23067 non-null  float64
 5   LotSizeAcres              23065 non-null  float64
 6   LotSizeArea               23070 non-null  float64
 7   YearBuilt                 23471 non-null  float64
 8   FireplaceYN               23469 non-null  object 
 9   LivingArea                23471 non-null  float64
 10  BathroomsTotalInteger     23483 non-null  float64
 11  Latitude                  23482 non-null  float64
 12  Longitude                 23482 non-null  float64
 13  ParkingTotal              23484 non-null  float64
 14  CloseDate  

### Approaches for Missing Value Handling

In general, there are several ways to handle missing values in data.

One method is to input the mean, median, or mode of the feature for the missing values. Another method is to remove the feature entirely, which makes more sense for features which are deemed not as important.

Features with complete or near-complete missingness are removed.

For features below roughly 10% missingness, mean, median, or mode imputation can alter relationships and reduce variance. Missing entries are instead filled with fixed-seed random draws from observed training values, preserving the empirical center and spread.

Held-out values are sampled only from training observations to prevent leakage.

#### Random Sample Imput Function

In [14]:
# random sampling impute function
def random_sample_impute(train_col: pd.Series, test_col: pd.Series) -> tuple[pd.Series, pd.Series]:

    nonnull = train_col.dropna()                                # get observed (non-missing) training values
    out_trn = train_col.copy()                                  # copy training column
    out_tst = test_col.copy()                                   # copy testing column

    n_missing_trn = out_trn.isna().sum()                        # count missing in train
    n_missing_tst = out_tst.isna().sum()                        # count missing in test

    sampled_trn = nonnull.sample(n_missing_trn, replace=True).values   # sample from observed train values
    sampled_tst = nonnull.sample(n_missing_tst, replace=True).values   # sample from observed train values

    out_trn.loc[out_trn.isna()] = sampled_trn                   # fill missing in train
    out_tst.loc[out_tst.isna()] = sampled_tst                   # fill missing in test

    return out_trn, out_tst                                     # return both columns

#### Random Sample Imput Loop

In [15]:
# reproducibility
np.random.seed(420)

# nan cols
nan_cols = ['AttachedGarageYN',
            'NewConstructionYN',
            'PoolPrivateYN',
            'GarageSpaces',
            'LotSizeSquareFeet',
            'LotSizeAcres',
            'LotSizeArea',
            'OriginalListPrice',
            'YearBuilt',
            'City',
            'FireplaceYN',
            'LivingArea',
            'BathroomsTotalInteger',
            'Longitude',
            'Latitude',
            'ParkingTotal',]

# random sampling impute loop
for col in nan_cols:
    trn[col], tst[col] = random_sample_impute(trn[col], tst[col])


In [16]:
# percentage missingness
print((trn.isna().mean() * 100).sort_values(ascending=False).head(10))
print((tst.isna().mean() * 100).sort_values(ascending=False).head(10))

AttachedGarageYN     0.0
PoolPrivateYN        0.0
NewConstructionYN    0.0
GarageSpaces         0.0
LotSizeSquareFeet    0.0
LotSizeAcres         0.0
LotSizeArea          0.0
YearBuilt            0.0
FireplaceYN          0.0
LivingArea           0.0
dtype: float64


AttachedGarageYN     0.0
PoolPrivateYN        0.0
NewConstructionYN    0.0
GarageSpaces         0.0
LotSizeSquareFeet    0.0
LotSizeAcres         0.0
LotSizeArea          0.0
YearBuilt            0.0
FireplaceYN          0.0
LivingArea           0.0
dtype: float64


## Incorrect Values



### Duplicates

Duplicate listing keys are resolved by retaining the record with the most recent closing date.

In [17]:
# check duplicate count
print(f"Duplicate count (trn): {trn['ListingKey'].duplicated().sum()}")

# drop duplicates based on most recent close date
trn["CloseDate"] = pd.to_datetime(trn["CloseDate"])  # ensure datetime format
trn = (trn
       .sort_values("CloseDate", ascending=False)    # newest first
       .drop_duplicates(subset="ListingKey", keep="first"))

# verify removal
print(f"Duplicate count after drop (trn): {trn['ListingKey'].duplicated().sum()}")

Duplicate count (trn): 57
Duplicate count after drop (trn): 0


The same duplicate-removal rule is applied to `tst`, preventing repeated properties from distorting held-out metrics.

In [18]:
# check duplicate count
print(f"Duplicate count (tst): {tst['ListingKey'].duplicated().sum()}")

# drop duplicates based on most recent close date
tst["CloseDate"] = pd.to_datetime(tst["CloseDate"])
tst = (tst
       .sort_values("CloseDate", ascending=False)
       .drop_duplicates(subset="ListingKey", keep="first"))

# verify removal
print(f"Duplicate count after drop (tst): {tst['ListingKey'].duplicated().sum()}")

Duplicate count (tst): 5
Duplicate count after drop (tst): 0


After deduplication, `ListingKey` and `CloseDate` are removed because they are no longer needed.

In [19]:
# drop listingkey and closedate from trn and tst
trn.drop('ListingKey', axis = 1, inplace = True)
tst.drop('ListingKey', axis = 1, inplace = True)
trn.drop('CloseDate', axis = 1, inplace = True)
tst.drop('CloseDate', axis = 1, inplace = True)

### Impossible Values

Impossible values are defined based on universal domain rules that apply to all residential properties. It is reasonable to assume that a true single-family residence:

- Must have more than 0 bedrooms.

- Must have more than 0 bathrooms.

- Must have more than 0 living area size (square feet).

- Must have more than 0 lot size, both in acres and in square feet.

- Must have 0 or more parking spaces (cannot be negative).

- Must have been built in the present year or earlier (not in the future).

- Must have been built after 1800, as listings with construction dates before this year are implausible or erroneous because most surviving pre-1800 residences are protected as historic monuments.

Additionally:

`DaysOnMarket` cannot be negative. Values equal to 0 are retained because, while rare, it is possible for a residence to be listed and closed on the same day.

For geographic consistency, only properties located within California are retained, defined as:
`Latitude` ∈ [32, 42], `Longitude` ∈ [−124, −114].

Records with `ClosePrice` equal to 0 are removed as impossible or highly irregular. The same rule is applied to `ListPrice` and `OriginalListPrice`, since prices cannot be zero in a true transaction.

The interior living area should not exceed the total lot size, i.e., `LivingArea ≤ LotSizeSquareFeet`. Properties violating this relationship are treated as erroneous.

`GarageSpaces` should not exceed `ParkingTotal`, since garages are a subset of total parking spaces.

IQR filtering made the discrete `GarageSpaces` and `ParkingTotal` features deterministic, so a domain-informed upper bound of 30 is used instead. More than 30 garage or parking spaces is highly unrealistic for a single-family residence; such records often represent estates.

In [20]:
# impossible value filters

# remove impossible or illogical property values
trn = trn[(trn['BedroomsTotal'] > 0) &
          (trn['BathroomsTotalInteger'] > 0) &
          (trn['LivingArea'] > 0) &
          (trn['LotSizeAcres'] > 0) &
          (trn['LotSizeSquareFeet'] > 0) &
          (trn['ParkingTotal'] >= 0) &
          (trn['YearBuilt'] <= pd.Timestamp.now().year) &
          (trn['YearBuilt'] >= 1800)]

tst = tst[(tst['BedroomsTotal'] > 0) &
          (tst['BathroomsTotalInteger'] > 0) &
          (tst['LivingArea'] > 0) &
          (tst['LotSizeAcres'] > 0) &
          (tst['LotSizeSquareFeet'] > 0) &
          (tst['ParkingTotal'] >= 0) &
          (tst['YearBuilt'] <= pd.Timestamp.now().year) &
          (tst['YearBuilt'] >= 1800)]

# remove negative days on market
trn = trn[trn['DaysOnMarket'] >= 0]

tst = tst[tst['DaysOnMarket'] >= 0]

# keep only properties within California lat/long bounds
trn = trn[(trn['Latitude'].between(32, 42)) &
          (trn['Longitude'].between(-124, -114))]

tst = tst[(tst['Latitude'].between(32, 42)) &
          (tst['Longitude'].between(-124, -114))]

# remove zero or impossible prices
trn = trn[(trn['ClosePrice'] > 0) &
          (trn['ListPrice'] > 0) &
          (trn['OriginalListPrice'] > 0)]

tst = tst[(tst['ClosePrice'] > 0) &
          (tst['ListPrice'] > 0) &
          (tst['OriginalListPrice'] > 0)]

# living area cannot exceed lot size
trn = trn[trn['LivingArea'] <= trn['LotSizeSquareFeet']]

tst = tst[tst['LivingArea'] <= tst['LotSizeSquareFeet']]

# garage spaces cannot exceed total parking
trn = trn[trn['GarageSpaces'] <= trn['ParkingTotal']]

tst = tst[tst['GarageSpaces'] <= tst['ParkingTotal']]

# garagespace and parking totall cannot exceed 30
trn = trn[(trn['GarageSpaces'] <= 30) &
          (trn['ParkingTotal'] <= 30)]

tst = tst[(tst['GarageSpaces'] <= 30) &
          (tst['ParkingTotal'] <= 30)]

## Outlier Treatment

### Percentile trimming

The final pipeline does not use the earlier IQR experiment. For selected numeric fields, it learns the 0.5th and 99.5th percentile bounds from training data only, retaining the middle 99% of each feature. The same fixed bounds are then applied to the held-out data.

This chronological, training-derived treatment limits extreme records without using information from September or October.

### Apply training-derived percentile bounds

In [21]:
# Numeric fields included in percentile trimming
num_cols = ['LotSizeSquareFeet',
            'LotSizeAcres',
            'LivingArea',
            'BathroomsTotalInteger',
            'BedroomsTotal',
            'ListPrice',
            'OriginalListPrice',
            'ClosePrice',
            'DaysOnMarket']

pct_bounds = {}
for col in num_cols:
    # Learn the 0.5th and 99.5th percentile thresholds from training data only.
    lower = trn[col].quantile(0.005)
    upper = trn[col].quantile(0.995)
    pct_bounds[col] = (lower, upper)
    trn = trn[(trn[col] >= lower) & (trn[col] <= upper)]

# Apply the fixed training thresholds to held-out data.
for col, (lower, upper) in pct_bounds.items():
    tst = tst[(tst[col] >= lower) & (tst[col] <= upper)]

# Exploratory Data Analysis

## Visualization

### Continuous Distributions

In [22]:
con_cols = trn.select_dtypes(include=['int64', 'float64']).columns

for col in con_cols:
    plt.figure(figsize=(6,4))
    sns.histplot(trn[col], kde=True, bins=30)
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.show()

C:\Users\mmiov\AppData\Local\Temp\ipykernel_84732\614953639.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\mmiov\AppData\Local\Temp\ipykernel_84732\614953639.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\mmiov\AppData\Local\Temp\ipykernel_84732\614953639.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\mmiov\AppData\Local\Temp\ipykernel_84732\614953639.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\mmiov\AppData\Local\Temp\ipykernel_84732\614953639.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


...

C:\Users\mmiov\AppData\Local\Temp\ipykernel_84732\614953639.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\mmiov\AppData\Local\Temp\ipykernel_84732\614953639.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\mmiov\AppData\Local\Temp\ipykernel_84732\614953639.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\mmiov\AppData\Local\Temp\ipykernel_84732\614953639.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\mmiov\AppData\Local\Temp\ipykernel_84732\614953639.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Boxplots after percentile trimming

These plots provide a visual check of the retained distributions after applying the training-derived bounds.

In [23]:
for col in num_cols:
    plt.figure(figsize=(6,3))
    sns.boxplot(x=trn[col])
    plt.title(f"Boxplot of {col}")
    plt.show()

C:\Users\mmiov\AppData\Local\Temp\ipykernel_84732\4219979734.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\mmiov\AppData\Local\Temp\ipykernel_84732\4219979734.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\mmiov\AppData\Local\Temp\ipykernel_84732\4219979734.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\mmiov\AppData\Local\Temp\ipykernel_84732\4219979734.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\mmiov\AppData\Local\Temp\ipykernel_84732\4219979734.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\mmiov\AppData\Local\Temp\ipykernel_84732\4219979734.py:2: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  plt.figure(figsize=(6,3))


C:\Users\mmiov\AppData\Local\Temp\ipykernel_84732\4219979734.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\mmiov\AppData\Local\Temp\ipykernel_84732\4219979734.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\mmiov\AppData\Local\Temp\ipykernel_84732\4219979734.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\mmiov\AppData\Local\Temp\ipykernel_84732\4219979734.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Categorical Distributions

In [24]:
cat_cols = trn.select_dtypes(include=['object', 'category', 'bool']).columns

for col in cat_cols:
    plt.figure(figsize=(6,3))
    trn[col].value_counts(normalize=True).plot(kind='bar')
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Proportion")
    plt.show()

C:\Users\mmiov\AppData\Local\Temp\ipykernel_84732\2206955664.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\mmiov\AppData\Local\Temp\ipykernel_84732\2206955664.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\mmiov\AppData\Local\Temp\ipykernel_84732\2206955664.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\mmiov\AppData\Local\Temp\ipykernel_84732\2206955664.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\mmiov\AppData\Local\Temp\ipykernel_84732\2206955664.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\mmiov\AppData\Local\Temp\ipykernel_84732\2206955664.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\mmiov\AppData\Local\Temp\ipykernel_84732\2206955664.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


`City` and `CountyOrParish` are excluded because their high cardinality would substantially expand the encoded feature space.

`ContractStatusChangeDate` is also excluded because it does not add necessary model information.

In [25]:
# drop
trn.drop(['City', 'CountyOrParish', 'ContractStatusChangeDate'], axis = 1, inplace = True)
tst.drop(['City', 'CountyOrParish', 'ContractStatusChangeDate'], axis = 1, inplace = True)

In [26]:
cat_cols = trn.select_dtypes(include=['object', 'category', 'bool']).columns

for col in cat_cols:
    print(f"\n=== {col} ===")
    print(trn[col].value_counts(dropna=False).head(10))
    print(f"Unique values: {trn[col].nunique()}")


=== AttachedGarageYN ===
AttachedGarageYN
True     65679
False    12495
Name: count, dtype: int64
Unique values: 2

=== PoolPrivateYN ===
PoolPrivateYN
False    64856
True     13318
Name: count, dtype: int64
Unique values: 2

=== NewConstructionYN ===
NewConstructionYN
False    75372
True      2802
Name: count, dtype: int64
Unique values: 2

=== FireplaceYN ===
FireplaceYN
True     57479
False    20695
Name: count, dtype: int64
Unique values: 2


### Summary Statistics and Distribution Diagnostics

In [27]:
numeric_cols = trn.select_dtypes(include=[np.number]).columns

summary = trn[numeric_cols].describe().T
summary['Skewness'] = trn[numeric_cols].skew()
summary['Kurtosis'] = trn[numeric_cols].kurt()
summary = summary[['mean','std','min','max','Skewness','Kurtosis']]
summary.round(2)

,mean,std,min,max,Skewness,Kurtosis
GarageSpaces,1.98,0.85,0.00,24.00,1.47,31.59
LotSizeSquareFeet,12901.74,23554.81,1980.00,438214.00,6.42,51.09
LotSizeAcres,0.29,0.53,0.05,5.60,6.14,43.93
LotSizeArea,11704.66,22801.25,0.00,1806869.00,18.26,1002.94
YearBuilt,1975.83,26.98,1801.00,2026.00,-0.16,-0.41
LivingArea,1979.10,803.86,675.00,6500.00,1.24,2.06
BathroomsTotalInteger,2.56,0.93,1.00,6.00,0.89,1.41
Latitude,34.73,1.68,32.12,41.89,1.24,0.29
Longitude,-118.63,1.85,-123.82,-114.35,-0.98,-0.46
ParkingTotal,2.75,1.91,0.00,30.00,3.38,22.42


### `ClosePrice`

#### Pairplot and Linear Correlations

In [28]:
corr = trn[numeric_cols].corr()

# Use a deterministic sample and a focused feature set so the pairplot remains readable.
pair_cols = ['ClosePrice', 'LivingArea', 'LotSizeAcres',
             'BedroomsTotal', 'BathroomsTotalInteger', 'DaysOnMarket']
plot_sample = trn[pair_cols].sample(n=min(3000, len(trn)), random_state=420)
sns.pairplot(plot_sample, corner=True, plot_kws={'alpha': 0.25, 's': 12})
plt.suptitle('Selected numeric feature relationships', y=1.02)
plt.show()

plt.figure(figsize=(12, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Correlation heatmap of numeric features')
plt.show()

C:\Users\mmiov\AppData\Local\Temp\ipykernel_84732\3462568659.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\mmiov\AppData\Local\Temp\ipykernel_84732\3462568659.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


#### Partial Correlation Coefficients

A partial corrrelation between `ClosePrice` and a predictor `X` measures how strongly they relate after accounting for the linear effects of all other predictors.

In [29]:
def partial_corr(df, x, y, covars):
    X_cov = df[covars]

    # regress x on covariates, get residuals
    model_x = LinearRegression().fit(X_cov, df[x])
    resid_x = df[x] - model_x.predict(X_cov)

    # regress y on covariates, get residuals
    model_y = LinearRegression().fit(X_cov, df[y])
    resid_y = df[y] - model_y.predict(X_cov)

    # correlation between residuals = partial correlation
    r, _ = pearsonr(resid_x, resid_y)
    return r

num_cols = trn.select_dtypes(include=['int64', 'float64']).columns
res = []
for col in num_cols:
    if col == 'ClosePrice':
        continue
    covars = [c for c in num_cols if c not in ['ClosePrice', col]]
    r = partial_corr(trn, x=col, y='ClosePrice', covars=covars)
    res.append({'Variable': col, 'Partial_r': r})

partial_corr_df = pd.DataFrame(res).sort_values(by='Partial_r', ascending=False)
print(partial_corr_df)

                 Variable  Partial_r
11              ListPrice   0.713381
12          BedroomsTotal   0.018860
7                Latitude   0.010435
6   BathroomsTotalInteger   0.003603
9            ParkingTotal  -0.002525
2            LotSizeAcres  -0.003119
3             LotSizeArea  -0.003548
0            GarageSpaces  -0.005223
1       LotSizeSquareFeet  -0.007467
5              LivingArea  -0.019882
13      OriginalListPrice  -0.028308
8               Longitude  -0.049475
4               YearBuilt  -0.073088
10           DaysOnMarket  -0.194543


#### Geographical Visualization

In [30]:
plt.figure(figsize=(8,8))
sns.scatterplot(data=trn, x='Longitude', y='Latitude',
                hue='ClosePrice', palette='rocket_r', alpha=0.5)
plt.title('California Price Distribution')

Text(0.5, 1.0, 'California Price Distribution')

#### `LogPrice = ln(ClosePrice)`

In [31]:
# Match the deployed model's natural-log target transformation.
trn['LogPrice'] = np.log(trn['ClosePrice'])
tst['LogPrice'] = np.log(tst['ClosePrice'])

sns.histplot(trn['LogPrice'], kde=True)
plt.title('Training distribution of natural-log sale price')
plt.show()

C:\Users\mmiov\AppData\Local\Temp\ipykernel_84732\3633636211.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Train and Test Compatibility

In [32]:
for col in num_cols:
    plt.figure(figsize=(8, 4))
    sns.kdeplot(trn[col], label='Train', fill=True)
    sns.kdeplot(tst[col], label='Test', fill=True)
    plt.title(f'{col}: Train vs Test')
    plt.legend(); plt.show()

C:\Users\mmiov\AppData\Local\Temp\ipykernel_84732\635592451.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.legend(); plt.show()


C:\Users\mmiov\AppData\Local\Temp\ipykernel_84732\635592451.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.legend(); plt.show()


C:\Users\mmiov\AppData\Local\Temp\ipykernel_84732\635592451.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.legend(); plt.show()


C:\Users\mmiov\AppData\Local\Temp\ipykernel_84732\635592451.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.legend(); plt.show()


C:\Users\mmiov\AppData\Local\Temp\ipykernel_84732\635592451.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.legend(); plt.show()


...

C:\Users\mmiov\AppData\Local\Temp\ipykernel_84732\635592451.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.legend(); plt.show()


C:\Users\mmiov\AppData\Local\Temp\ipykernel_84732\635592451.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.legend(); plt.show()


C:\Users\mmiov\AppData\Local\Temp\ipykernel_84732\635592451.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.legend(); plt.show()


C:\Users\mmiov\AppData\Local\Temp\ipykernel_84732\635592451.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.legend(); plt.show()
C:\Users\mmiov\AppData\Local\Temp\ipykernel_84732\635592451.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.legend(); plt.show()


C:\Users\mmiov\AppData\Local\Temp\ipykernel_84732\635592451.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.legend(); plt.show()


### Reproducibility boundary

Raw and processed MLS records are intentionally not exported or included in this repository. The executed outputs preserve aggregate evidence while protecting listing-level data.